# 03. 서연 — 공통 구현 재현과 기존 보고서 대조

공통 노트북 02의 환경 설정·개인/집단 비교 절차를 바탕으로 만든 독립 실험입니다. `src/`, 공통 01·02는 변경하지 않습니다.
작업 브랜치: `feat/seoyeon-tuning`. 참조: main `07d88da`, 승호 `1806f0a`.

공통 노트북 원본 실행 결과는 `results/seoyeon_tuning/common_replay/`에 별도로 보존됩니다. 이 노트북은 전처리 자료량과 공통 유효 표본의 경보 분포를 재계산하고, 승호 코드의 경보선 정의를 실측 대조합니다.
기존 결과보고서의 70%·40/30/30 결과는 과거 실행값으로 인용하며, 이번 공통 구현 결과와 동일 실험으로 간주하지 않습니다.

In [ ]:
import sys
from pathlib import Path
from datetime import datetime, timezone
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src import evaluation as ev
from experiments.seoyeon import audit
WHO = '조서연'
SAVE = True  # 이 검증 결과를 로컬 results/results.csv에 남깁니다.


## 실제 실행 범위와 해석

- 착용률 80%, 수면 조각 간격 90분, 최소 관측 42일, 개인 과거 28달력일/최소 14일.
- 걸음·수면·취침 각각에서 개인/집단 점수가 모두 유효한 동일 인일만 비교합니다.
- 목표 경보율 1/30과 과거 조원 자료의 5%를 각각 재계산합니다. 동률 때문에 목표 경보율과 실제 경보율이 다를 수 있습니다.
- 승호 브랜치는 사람 개발70%/확인30%입니다. 확인용 자료에서 보정한 `h`와 평가기 분위수 경보선을 대조합니다.
- 승호 주입 진단은 걸음 −30%, 14일, seed 0, **단 한 번의 판정선 영향 점검**입니다. 전체 탐색 또는 최종 20회 평가의 재현이 아닙니다.
- 무라벨 자료의 경보율·겹침·집중도는 정확도나 임상 오탐률이 아닙니다.


In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
OUT = ROOT / 'results' / 'seoyeon_tuning' / ('audit_' + stamp)
result = audit.run(OUT)
print('실제 실행 완료:', OUT.relative_to(ROOT))
display(result['counts'])
display(pd.Series(result['evidence']['quality'], name='값').to_frame())
display(pd.Series(result['evidence']['reference'], name='값').to_frame())


In [ ]:
display(result['comparisons'])
display(result['cusum_thresholds'])
display(result['fixed_rules'])
display(Image(filename=str(OUT / 'alarm_comparison.png')))


## 그래프 해석

왼쪽은 경보가 없었던 사람 수, 가운데는 경보 수 상위 10% 사람이 차지한 몫, 오른쪽은 실제 경보 인일/유효 인일입니다. 모두 목표 1/30, 요일 보정 없음 조건입니다. 지표별 공통 인일과 사람 수는 CSV에 기록됩니다.

집단 경보의 집중도가 높다는 사실만으로 해당 경보가 틀렸다거나 개인 기준선이 정확하다고 결론 내릴 수 없습니다. 경보선 대조표의 `disagreement_days`는 주입 없는 자료, `injected_disagreement_days`는 seed 0 주입 구간의 판정 불일치입니다.


In [ ]:
if SAVE:
    for row in result['comparisons'].to_dict('records'):
        ev.log_result(WHO, 'NetHealth', '공통표본 개인-집단 재현',
                      f"wear=80, window=28, feature={row['feature']}, dow={row['dow_adjust']}, rate={row['target_rate']}", row)
    print('검증된 집계 결과 12개를 results/results.csv에 기록했습니다.')
else:
    print('SAVE=False: 공용 결과표에는 기록하지 않습니다.')


## 기존 보고서와 비교할 때의 조건 차이

|항목|기존 보고서 주 기준|저장소 main|승호 최종 코드|
|---|---|---|---|
|착용 기준|70%|80%|80%|
|분할|시간순 개발40/보정30/시험30|전체 자료 후향적 보정|사람 개발70/확인30|
|수면|유효·비중첩 조각 합산, 최장 에피소드|모든 조각 합산, 90분 이내 묶음|main 전처리|
|특징 결합|걸음·수면·취침 최대 절댓값|지표별 별도 평가|최종은 걸음만|
|집단|개발 기간 인일 통합|같은 날 최소30명|같은 날 개발+확인용|
|경보 기준|별도 보정기간 95백분위수|전체 분포 1/30|확인용 분포 1/30|
|주 검증|무라벨 시험 경보 분포|무라벨 비교+인공 변화 주입|개발 탐색 후 인공 변화 주입|

기존 6모델 결과는 `existing_primary40_summary.csv`에 과거 실행값으로 복사됩니다. 이번에는 해당 모델들과 MOMENT를 다시 학습/추론하지 않았습니다. 상세 코드 검토는 `experiments/seoyeon/REVIEW.md`에 있습니다.


In [ ]:
print('기존 실행값 인용: 이번 재실행 결과 아님')
display(pd.read_csv(OUT / 'existing_primary40_summary.csv'))
print('기존 보고서와 원자료 해시 보존:', result['evidence']['existing_report_preserved'])
